# Heo 2024 vs Sheen 2018 local magnitude — event-by-event comparison

Both scales use the **same** catalogue, the **same** detectability gate (`require_pick`) and the
**same** SNR threshold (3.0); they differ only in the published calibration and the components used:

| | components | distance | reference | constant |
|---|---|---|---|---|
| **Heo 2024**  | vertical only           | hypocentral | 17 km  | +2.0 |
| **Sheen 2018**| all 3 (geom-mean H + Z) | epicentral  | 100 km | +3.0 |

Questions: do they cover the **same events**, and **why** do the magnitudes differ?

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from seismostats.analysis import estimate_mc_maxc, ClassicBValueEstimator
from seismostats.utils import bin_to_precision
DM = 0.1
h = pd.read_csv("catalog_phasenet_plus_2010_2024_blastclean_with_ml_heo.csv"); s = pd.read_csv("catalog_phasenet_plus_2010_2024_blastclean_with_ml_sheen.csv")
for d in (h, s): d["time"] = pd.to_datetime(d["time"])
m = h.merge(s, on="time", suffixes=("_heo", "_sheen"))
both = m.dropna(subset=["magnitude_heo", "magnitude_sheen"]).copy()
both["d"] = both.magnitude_sheen - both.magnitude_heo
print(f"Heo {h.magnitude.notna().sum():,} | Sheen {s.magnitude.notna().sum():,} | both {len(both):,} events with ML")

## 1. Same number of estimates?

**Yes** — identical event set with a magnitude (shared SNR gate + `require_pick`). Only the
per-event channel count differs: Heo uses each picked station's vertical; Sheen uses all 3 components.

In [ ]:
n_h, n_s = m.magnitude_heo.notna().sum(), m.magnitude_sheen.notna().sum()
print(f"events with ML — Heo {n_h:,} | Sheen {n_s:,} | identical: {n_h == n_s}")
print(f"n_used/event   — Heo median {m.n_used_heo.median():.0f} | Sheen median {m.n_used_sheen.median():.0f} (~3x: all comps)")
fig, ax = plt.subplots(figsize=(7, 3.0), dpi=120)
ax.hist(m.n_used_heo.dropna(), bins=range(0, 40), alpha=0.6, color="tab:green", label="Heo (Z)")
ax.hist(m.n_used_sheen.dropna(), bins=range(0, 40), alpha=0.6, color="tab:purple", label="Sheen (3-comp)")
ax.set_xlabel("n_used (station-channels)"); ax.set_ylabel("events"); ax.legend()
ax.set_title("Same events, different channel count"); fig.tight_layout(); plt.show()

## 2. Event-by-event scatter

A tight linear relation, but with **slope < 1** (not a parallel shift): Sheen reads higher and the
gap shrinks toward large magnitudes.

In [ ]:
d = both.d
print(f"Sheen - Heo: median {d.median():+.2f}  mean {d.mean():+.2f}  std {d.std():.2f}")
SLOPE, INT = np.polyfit(both.magnitude_heo, both.magnitude_sheen, 1)
print(f"fit: Sheen = {INT:+.2f} + {SLOPE:.2f}*Heo   (slope < 1)")
fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 4.4), dpi=120)
lim = [both.magnitude_heo.min()-0.3, both.magnitude_sheen.max()+0.3]
a0.hexbin(both.magnitude_heo, both.magnitude_sheen, gridsize=45, cmap="viridis", mincnt=1)
a0.plot(lim, lim, "k--", lw=1, label="1:1")
a0.plot(lim, [INT+SLOPE*x for x in lim], "r-", lw=1.4, label=f"fit ({SLOPE:.2f}*Heo{INT:+.2f})")
a0.set_xlim(lim); a0.set_ylim(lim); a0.set_xlabel("Heo 2024 ML"); a0.set_ylabel("Sheen 2018 ML")
a0.set_title("Event ML: Sheen vs Heo"); a0.legend(fontsize=8)
a1.hist(d, bins=np.arange(-0.5, 2.0, 0.05), color="0.5")
a1.axvline(d.median(), color="r", lw=1.5, label=f"median {d.median():+.2f}")
a1.set_xlabel("Sheen - Heo (ML)"); a1.set_ylabel("events"); a1.set_title("Offset distribution"); a1.legend(fontsize=8)
fig.tight_layout(); plt.show()

## 3. Why they differ — and why it sets the b-value

The fit **Sheen ≈ 0.96 + 0.59·Heo** has a **slope of 0.59, not 1**. Two separable effects:

- **Level (~+0.8 at small M): the component basis.** Sheen's event ML is the median over all
  three components, dominated by the larger **horizontal** S amplitudes (+ horizontal coefficients);
  Heo is **vertical-only**. The distance laws are nearly identical for the same amplitude/distance —
  the formula offset `Sheen_Z(R) − Heo(R)` runs ~+0.26→−0.1 over 30–150 km (≈0, left panel) — so the
  level shift is the component basis (H vs Z), **not** the −logA₀ functions.
- **Slope (0.59): sets the b-value.** Heo's steeper distance term (1.45·log₁₀(R/17) vs Sheen's
  0.51·log₁₀(R/100)) stretches Heo's magnitude range, so a unit of Heo maps to 0.59 of Sheen. A
  non-unit slope **directly fixes the b-value ratio**: `b_Sheen = b_Heo / 0.59 ≈ 1.7×` (confirmed in §4).

In [ ]:
def sheenZ(R): return 0.5107*np.log10(R/100) + 0.001699*(R-100) + 3.0
def heo(R):    return 1.450676*np.log10(R/17) - 0.000661*(R-17) + 2.0
R = np.linspace(5, 200, 200)
fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 4.2), dpi=120)
a0.plot(R, sheenZ(R) - heo(R), color="tab:red", lw=2)
a0.axhline(0, color="0.6", lw=0.7, ls=":")
a0.axhline(both.d.median(), color="tab:blue", lw=1.2, ls="--", label=f"empirical median {both.d.median():+.2f}")
a0.set_xlabel("distance R (km)"); a0.set_ylabel("ΔML = Sheen$_Z$ − Heo (same A)")
a0.set_title("Formula offset ≈ 0 (distance law is NOT the cause)"); a0.legend(fontsize=8)
a1.hexbin(both.magnitude_heo, both.d, gridsize=40, cmap="magma", mincnt=1)
a1.plot(np.sort(both.magnitude_heo), INT + (SLOPE-1)*np.sort(both.magnitude_heo), "c-", lw=1.4,
        label=f"from fit (slope {SLOPE:.2f})")
a1.set_xlabel("Heo ML"); a1.set_ylabel("Sheen − Heo")
a1.set_title("Offset shrinks with M (slope ≠ 1 → different b)"); a1.legend(fontsize=8)
fig.tight_layout(); plt.show()
print("level (+0.8 at small M): component basis (horizontal vs vertical amplitude)")
print(f"slope ({SLOPE:.2f}): different distance steepness  =>  b_Sheen = b_Heo / {SLOPE:.2f}")

## 4. FMD side by side — the b-value link

Same events, related by a slope-0.59 mapping → the Heo FMD sits ~0.8 lower in magnitude, and its
b-value is lower by the same slope factor.

In [ ]:
from seismostats.analysis import estimate_mc_ks   # K-S Mc (MAXC + b-estimator imported above)

def _mc_b_both(mags):
    mm = bin_to_precision(np.sort(mags.astype(float)), DM)
    mc_mx, _ = estimate_mc_maxc(mm, fmd_bin=DM)
    be = ClassicBValueEstimator(); be.calculate(mm[mm >= mc_mx], mc=mc_mx, delta_m=DM)
    b_mx, bse_mx = be.b_value, be.std
    try:
        out = estimate_mc_ks(mm, delta_m=DM, p_value_pass=0.1)
        mc_ks = out[0] if isinstance(out, tuple) else out
    except Exception:
        mc_ks = None
    b_ks = bse_ks = None
    if mc_ks is not None and np.isfinite(mc_ks) and (mm >= mc_ks).sum() >= 30:
        be2 = ClassicBValueEstimator(); be2.calculate(mm[mm >= mc_ks], mc=mc_ks, delta_m=DM)
        b_ks, bse_ks = be2.b_value, be2.std
    return dict(mc_mx=mc_mx, b_mx=b_mx, bse_mx=bse_mx, mc_ks=mc_ks, b_ks=b_ks, bse_ks=bse_ks)

fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), dpi=120, sharey=True)
res = {}
for a, mags, name, col in [(axs[0], both.magnitude_heo, "Heo 2024", "tab:green"),
                           (axs[1], both.magnitude_sheen, "Sheen 2018", "tab:purple")]:
    r = _mc_b_both(mags.to_numpy()); res[name] = r
    mm = bin_to_precision(np.sort(mags.to_numpy().astype(float)), DM)
    edges = np.arange(np.floor(mags.min()/DM)*DM, np.ceil(mags.max()/DM)*DM + DM, DM)
    c, _ = np.histogram(mags, bins=edges); cum = np.cumsum(c[::-1])[::-1]
    a.semilogy(edges[:-1], cum, ".", color=col, ms=4)
    # MAXC Mc + Gutenberg-Richter b-slope (red dashed)
    a.axvline(r["mc_mx"], color="r", ls="--", lw=0.9)
    amx = np.log10((mm >= r["mc_mx"]).sum()) + r["b_mx"] * r["mc_mx"]
    xmx = np.linspace(r["mc_mx"], mm.max(), 30)
    a.semilogy(xmx, 10**(amx - r["b_mx"] * xmx), "r--", lw=1.4,
               label=f"MAXC  Mc={r['mc_mx']:.2f}  b={r['b_mx']:.2f}\u00b1{r['bse_mx']:.2f}")
    # K-S Mc + b-slope (blue dotted)
    if r["b_ks"] is not None:
        a.axvline(r["mc_ks"], color="tab:blue", ls=":", lw=0.9)
        aks = np.log10((mm >= r["mc_ks"]).sum()) + r["b_ks"] * r["mc_ks"]
        xks = np.linspace(r["mc_ks"], mm.max(), 30)
        a.semilogy(xks, 10**(aks - r["b_ks"] * xks), color="tab:blue", ls=":", lw=1.4,
                   label=f"K-S  Mc={r['mc_ks']:.2f}  b={r['b_ks']:.2f}\u00b1{r['bse_ks']:.2f}")
    a.set_title(name); a.set_xlabel("Local magnitude"); a.legend(fontsize=8, loc="upper right")
axs[0].set_ylabel("N (\u2265 M)"); fig.tight_layout(); plt.show()
b_h = res["Heo 2024"]["b_mx"]; b_s = res["Sheen 2018"]["b_mx"]
print(f"MAXC:  b_Heo={b_h:.2f}  b_Sheen={b_s:.2f}  |  b_Heo/slope = {b_h/SLOPE:.2f}  (predicts b_Sheen)")
if res["Heo 2024"]["b_ks"] is not None and res["Sheen 2018"]["b_ks"] is not None:
    bkh, bks = res["Heo 2024"]["b_ks"], res["Sheen 2018"]["b_ks"]
    print(f"K-S :  b_Heo={bkh:.2f}  b_Sheen={bks:.2f}  |  b_Heo/slope = {bkh/SLOPE:.2f}  (predicts b_Sheen)")

## Conclusion

- **Same number of estimates** — same events with ML (shared SNR gate + `require_pick`); only the
  channel count differs (Heo vertical-only ≈ 5, Sheen all-3-component ≈ 15).
- The two scales are related by **Sheen ≈ 0.96 + 0.59·Heo**. The **+0.8 level** at small magnitudes is
  the **component basis** (horizontal-dominated 3-component median vs vertical-only), **not** the
  distance attenuation (which is nearly identical). The **0.59 slope** — from Heo's steeper distance
  term — **sets the b-value**: b_Sheen = b_Heo / 0.59 ≈ 1.7× (0.77 → 1.31).
- For this **vertical-dominated, dense micro-earthquake** catalogue, **Heo 2024** is the internally
  consistent representative ML (lower Mc, cleaner FMD); Sheen is a useful all-Korea cross-check.